In [1]:
import joblib
import pandas as pd
import sys
from pathlib import Path

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.config import ARTIFACTS_DIR, DEFAULT_RUN_TAG, get_run_artifact_dirs

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)

DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]
DEPLOY_MODELS_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_models.joblib"
DEPLOY_MANIFEST_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_manifest.csv"

FEATURE_ANALYSIS_ARTIFACT_DIR = ARTIFACTS_DIR / "feature_analysis" / RUN_TAG
FEATURE_ANALYSIS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PERMUTATION_IMPORTANCE_PATH = FEATURE_ANALYSIS_ARTIFACT_DIR / "permutation_importance.csv"
FEATURE_ANALYSIS_META_PATH = FEATURE_ANALYSIS_ARTIFACT_DIR / "feature_analysis_meta.json"

deploy_bundle = joblib.load(DEPLOY_MODELS_PATH)
deploy_manifest = pd.read_csv(DEPLOY_MANIFEST_PATH)

deployed_models = {
    deploy_id: {
        "model": payload["model"],
        "model_name": payload["model_name"],
        "profile": payload["profile"],
        "threshold": float(payload["threshold"]),
    }
    for deploy_id, payload in deploy_bundle.items()
}

deployed_model_registry_df = pd.DataFrame(
    [
        {
            "deploy_id": deploy_id,
            "model_name": spec["model_name"],
            "profile": spec["profile"],
            "threshold": spec["threshold"],
        }
        for deploy_id, spec in deployed_models.items()
    ]
).sort_values("deploy_id").reset_index(drop=True)

print(f"Feature-analysis artifacts: {FEATURE_ANALYSIS_ARTIFACT_DIR}")
deployed_model_registry_df

Feature-analysis artifacts: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1


,deploy_id,model_name,profile,threshold
0,deploy_f2,knn,f2,0.363636
1,deploy_precision,hgb,precision_constrained,0.885432
2,deploy_recall,rf,recall_constrained,0.029039


In [3]:
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
X_koi_test = KOI_test_set.drop(columns=["label", "group_id"]).copy()
y_koi_test = KOI_test_set["label"].copy()
groups_koi_test = KOI_test_set["group_id"].copy()

X_k2p = K2P_set.drop(columns=["label", "group_id"]).copy()
y_k2p = K2P_set["label"].copy()
groups_k2p = K2P_set["group_id"].copy()

X_combined = pd.concat([X_koi_test, X_k2p], axis=0, ignore_index=True)
y_combined = pd.concat([y_koi_test, y_k2p], axis=0, ignore_index=True)
groups_combined = pd.concat([groups_koi_test, groups_k2p], axis=0, ignore_index=True)

evaluation_sets = {
    "KOI_test": (X_koi_test, y_koi_test),
    "K2P": (X_k2p, y_k2p),
    "KOI_test_plus_K2P": (X_combined, y_combined),
}